# 1. Introducción

**Problema industrial:** Detección temprana de degradación mecánica por vibración.

**Activo analizado:** PUMP101 — sensores de vibración RMS y pico.

**Origen de datos:** Tendencias de vibración exportadas desde PI cada hora.

**Objetivo del análisis:** Evaluar degradación y comparar con umbrales ISO 10816 (simulados).


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Tendencia de vibración RMS y clasificación por zona ISO.

In [ ]:
tag_vib = "PUMP101.VIBRATION_RMS"
umbrales = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Umbrales")
umbral_alerta = float(umbrales.loc[umbrales["Tag"] == tag_vib, "Alerta"].iloc[0])
umbral_critico = float(umbrales.loc[umbrales["Tag"] == tag_vib, "Critico"].iloc[0])

vib = df_good[df_good["Tag"] == tag_vib].set_index("Timestamp")["Value"].sort_index()
vib_d = vib.resample("1D").mean()
pendiente = np.polyfit(np.arange(len(vib_d)), vib_d.values, 1)[0]

def clasificar_zona(val):
    if val < umbral_alerta:
        return "Aceptable"
    if val < umbral_critico:
        return "Alerta"
    return "Critico"

zona_actual = clasificar_zona(vib.iloc[-1])
resultados_export = pd.DataFrame({
    "Metrica": ["Vib_Actual", "Pendiente_diaria", "Umbral_Alerta", "Umbral_Critico", "Zona"],
    "Valor": [vib.iloc[-1], pendiente, umbral_alerta, umbral_critico, zona_actual],
})
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(vib.index, vib.values, color="steelblue", alpha=0.6, label="RMS horario")
axes[0].plot(vib_d.index, vib_d.values, color="navy", linewidth=2, label="Promedio diario")
axes[0].axhline(umbral_alerta, color="orange", linestyle="--", label="Alerta")
axes[0].axhline(umbral_critico, color="red", linestyle="--", label="Crítico")
axes[0].set_ylabel("mm/s")
axes[0].set_title("Tendencia de vibración PUMP101")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mapa de calor: vibración por día y hora
vib_h = vib.to_frame("RMS")
vib_h["Dia"] = vib_h.index.dayofyear
vib_h["Hora"] = vib_h.index.hour
pivot = vib_h.pivot_table(index="Hora", columns="Dia", values="RMS", aggfunc="mean")
im = axes[1].imshow(pivot.values, aspect="auto", cmap="YlOrRd")
axes[1].set_title("Mapa de calor vibración (hora vs. día)")
axes[1].set_xlabel("Día del año")
axes[1].set_ylabel("Hora")
plt.colorbar(im, ax=axes[1], label="mm/s")


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Vibracion", index=False)
    vib_d.reset_index().to_excel(writer, sheet_name="Serie_Diaria", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

El incremento sostenido de vibración RMS durante los últimos 30 días indica posible degradación del rodamiento. Se recomienda inspección predictiva (análisis de espectro) antes de la falla funcional.
